In [14]:
import pandas as pd
from pathlib import Path

base_path = Path(r'C:\Taxi')

# Read a local Parquet file
file_path = base_path / 'yellow_tripdata_2025-01.parquet'
df = pd.read_parquet(file_path)

# Display the first few rows
print(df.shape)
df.head()

(3475226, 20)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [15]:
for col in df.columns:
    null_count = df[col].isnull().sum()
    null_count_percent = df.isna().mean() *100
    if null_count >0 :
        print(f"{col}: nulls={null_count}")

passenger_count: nulls=540149
RatecodeID: nulls=540149
store_and_fwd_flag: nulls=540149
congestion_surcharge: nulls=540149
Airport_fee: nulls=540149


In [16]:
null_cols = ['passenger_count', 'RatecodeID', 'store_and_fwd_flag',
             'congestion_surcharge', 'Airport_fee']

rows_with_nulls = df[null_cols].isnull().any(axis=1)
df = df.dropna(subset=null_cols, how='all')
print(rows_with_nulls.sum())
print(df.shape)

540149
(2935077, 20)


In [17]:
df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

All of the columns null values are from the same rows. We will drop all of those rows. The intial shpae for the dataframe was (3475226, 20) and after dropping 540149 rows the final shape of the dataframe with no null values is  :(2935077, 20) which make sense as well.

Now first we will seperate the date and the time which are combined in the same column for the pick up and drop off date & time. And then will drop the original column as well.

In [18]:
import pandas as pd

# Clean, optimized loop using native Pandas datatypes
for col in ['tpep_pickup_datetime', 'tpep_dropoff_datetime']:
    if col in df.columns:
        # Convert original column to datetime first for stability
        df[col] = pd.to_datetime(df[col])
        
        # Determine prefix dynamically (Pickup or Dropoff)
        prefix = "Pickup" if col == 'tpep_pickup_datetime' else "Dropoff"
        
        # 1. Keeps 'datetime64[ns]' type (Time zeroes out to 00:00:00)
        df[f"{prefix}_date"] = df[col].dt.normalize()
        
        # 2. Converts to 'timedelta64[ns]' (Duration since midnight)
        df[f"{prefix}_time"] = pd.to_timedelta(df[col].dt.time.astype(str))

df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,Pickup_date,Pickup_time,Dropoff_date,Dropoff_time
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,0.0,1.0,18.00,2.5,0.0,0.0,2025-01-01,0 days 00:18:38,2025-01-01,0 days 00:26:59
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,0.0,1.0,12.12,2.5,0.0,0.0,2025-01-01,0 days 00:32:40,2025-01-01,0 days 00:35:13
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,0.0,1.0,12.10,2.5,0.0,0.0,2025-01-01,0 days 00:44:04,2025-01-01,0 days 00:46:01
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.0,1.0,9.70,0.0,0.0,0.0,2025-01-01,0 days 00:14:27,2025-01-01,0 days 00:20:01
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.0,1.0,8.30,0.0,0.0,0.0,2025-01-01,0 days 00:21:34,2025-01-01,0 days 00:25:06


In [20]:
#Drop the rows whose dropoff location is ealrier than the pickup location
mask = df['tpep_dropoff_datetime'] < df['tpep_pickup_datetime']
invalid_rows = df[mask].index
print(df.shape)
df =df.drop(invalid_rows)
print("New shape",df.shape)

(2935071, 24)
New shape (2935071, 24)


In [21]:
new_df= df.drop(['tpep_dropoff_datetime','tpep_pickup_datetime'],axis=1)

We have added four new column and dropped 2 column . Hence the new shape of the dataframe is (2935077,22)

In [22]:
new_df.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,Pickup_date,Pickup_time,Dropoff_date,Dropoff_time
count,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,...,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2.935071e+06,2935071,2935071,2935071,2935071
mean,1.775327e+00,1.297860e+00,3.194120e+00,2.482371e+00,1.668033e+02,1.659061e+02,1.227396e+00,1.754081e+01,1.552772e+00,4.741675e-01,...,4.995782e-01,9.467434e-01,2.662824e+01,2.225241e+00,1.239113e-01,4.734008e-01,2025-01-16 11:16:20.435976,0 days 14:52:26.652742,2025-01-16 11:26:35.877582,0 days 14:57:09.603903
min,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-9.000000e+02,-7.500000e+00,-5.000000e-01,...,-1.269400e+02,-1.000000e+00,-9.010000e+02,-2.500000e+00,-1.750000e+00,-7.500000e-01,2024-12-31 00:00:00,0 days 00:00:00,2024-12-31 00:00:00,0 days 00:00:00
25%,2.000000e+00,1.000000e+00,9.600000e-01,1.000000e+00,1.320000e+02,1.160000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,...,0.000000e+00,1.000000e+00,1.554000e+01,2.500000e+00,0.000000e+00,0.000000e+00,2025-01-09 00:00:00,0 days 11:20:24,2025-01-09 00:00:00,0 days 11:26:46
50%,2.000000e+00,1.000000e+00,1.600000e+00,1.000000e+00,1.620000e+02,1.620000e+02,1.000000e+00,1.210000e+01,1.000000e+00,5.000000e-01,...,0.000000e+00,1.000000e+00,2.018000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2025-01-16 00:00:00,0 days 15:33:23,2025-01-16 00:00:00,0 days 15:43:24
75%,2.000000e+00,1.000000e+00,2.990000e+00,1.000000e+00,2.340000e+02,2.340000e+02,1.000000e+00,1.910000e+01,2.500000e+00,5.000000e-01,...,0.000000e+00,1.000000e+00,2.820000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2025-01-24 00:00:00,0 days 19:05:29,2025-01-24 00:00:00,0 days 19:13:07
max,7.000000e+00,9.000000e+00,4.473030e+04,9.900000e+01,2.650000e+02,2.650000e+02,5.000000e+00,8.633721e+05,1.500000e+01,6.500000e+00,...,1.709400e+02,1.000000e+00,8.633804e+05,2.500000e+00,6.750000e+00,7.500000e-01,2025-02-01 00:00:00,0 days 23:59:59,2025-02-01 00:00:00,0 days 23:59:59
std,4.318818e-01,7.507506e-01,4.261892e+01,1.163210e+01,6.296516e+01,6.888156e+01,5.901153e-01,5.042899e+02,1.928508e+00,1.484049e-01,...,2.115801e+00,3.016248e-01,5.044850e+02,9.039895e-01,4.725094e-01,3.653121e-01,NaN,0 days 05:32:29.369658,NaN,0 days 05:37:42.272827


In [23]:
import pandas as pd

# Use the cleaned dataframe if available; otherwise fall back to the original one
analysis_df = new_df if 'new_df' in locals() else df

report = pd.DataFrame({
    'column': analysis_df.columns,
    'dtype': analysis_df.dtypes.astype(str),
    'missing_count': analysis_df.isna().sum(),
    'missing_pct': (analysis_df.isna().mean() * 100).round(2),
    'unique_count': analysis_df.nunique(dropna=False),
})

print('Dataset shape:', analysis_df.shape)
print('Duplicate rows:', analysis_df.duplicated().sum())
print('\nMissing values summary:')
print(report[report['missing_count'] > 0].sort_values('missing_count', ascending=False))

print('\nColumns with no missing values:')
print(report[report['missing_count'] == 0]['column'].tolist())

report.to_csv('data_quality_report.csv', index=False)
print('\nSaved report to data_quality_report.csv')

Dataset shape: (2935071, 22)
Duplicate rows: 0

Missing values summary:
Empty DataFrame
Columns: [column, dtype, missing_count, missing_pct, unique_count]
Index: []

Columns with no missing values:
['VendorID', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'Pickup_date', 'Pickup_time', 'Dropoff_date', 'Dropoff_time']

Saved report to data_quality_report.csv


In [24]:
negative_columns = []

for col in new_df.columns:
    if pd.api.types.is_numeric_dtype(new_df[col]):
        if (new_df[col] < 0).any():
            negative_columns.append(col)

print(negative_columns)

['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


In [26]:
print("Shape of the dataframe before performing any operation:", new_df.shape)

# Count negative trip distances without relying on positional label access
negative_trip_distance_mask = new_df["trip_distance"] < 0
negative_trip_distance_count = int(negative_trip_distance_mask.sum())
print("The number of rows with the negative trip_distance:", negative_trip_distance_count)

# Refund amount
# 1. Define the target financial columns
financial_cols = [
    'fare_amount', 'extra', 'mta_tax', 'tolls_amount',
    'improvement_surcharge', 'total_amount', 'congestion_surcharge',
    'airport_fee', 'cbd_congestion_fee'
]

# 2. Find rows where ANY of these columns have a value less than 0
# and trip_distance is positive.
existing_cols = [col for col in financial_cols if col in new_df.columns]

if existing_cols:
    indices_to_drop = new_df.index[
        (new_df[existing_cols].lt(0).any(axis=1)) &
        (new_df['trip_distance'] > 0)
    ]

    # 3. Drop the rows from your DataFrame
    new_df = new_df.drop(index=indices_to_drop)
    print(f"Dropped {len(indices_to_drop)} rows with negative financial values.")
else:
    print("No financial columns found in the dataframe.")

# Verify if the pickup and dropoff IDs are located inside the lookup table.
from pathlib import Path

base_path = Path(r'C:\Taxi')
zone_data = pd.read_csv(base_path / 'taxi_zone_lookup.csv')

# Find rows where BOTH locations are missing from zone_data
invalid_location_mask = (
    ~new_df['PULocationID'].isin(zone_data['LocationID']) &
    ~new_df['DOLocationID'].isin(zone_data['LocationID'])
)
invalid_location_ids = new_df.index[invalid_location_mask]
print(f"Rows with both locations missing from lookup table: {len(invalid_location_ids)}")
print("Updated shape:", new_df.shape)


Shape of the dataframe before performing any operation: (2935071, 22)
The number of rows with the negative trip_distance: 0
Dropped 55127 rows with negative financial values.
Rows with both locations missing from lookup table: 0
Updated shape: (2879944, 22)


In [27]:
new_df.describe()

,VendorID,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,Pickup_date,Pickup_time,Dropoff_date,Dropoff_time
count,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,...,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2.879944e+06,2879944,2879944,2879944,2879944
mean,1.771026e+00,1.296422e+00,3.181110e+00,2.507165e+00,1.669460e+02,1.660585e+02,1.185812e+00,1.828931e+01,1.601988e+00,4.925050e-01,...,5.224205e-01,9.839879e-01,2.766460e+01,2.307611e+00,1.321440e-01,4.840852e-01,2025-01-16 11:33:48.629445,0 days 14:52:20.571854,2025-01-16 11:43:59.651331,0 days 14:57:08.178177
min,1.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,-8.500000e+02,-7.500000e+00,-5.000000e-01,...,-2.300000e+01,-1.000000e+00,-8.510000e+02,-2.500000e+00,-1.750000e+00,-7.500000e-01,2024-12-31 00:00:00,0 days 00:00:00,2024-12-31 00:00:00,0 days 00:00:00
25%,2.000000e+00,1.000000e+00,9.600000e-01,1.000000e+00,1.320000e+02,1.160000e+02,1.000000e+00,8.600000e+00,0.000000e+00,5.000000e-01,...,0.000000e+00,1.000000e+00,1.581000e+01,2.500000e+00,0.000000e+00,0.000000e+00,2025-01-09 00:00:00,0 days 11:20:04,2025-01-09 00:00:00,0 days 11:26:31
50%,2.000000e+00,1.000000e+00,1.600000e+00,1.000000e+00,1.620000e+02,1.620000e+02,1.000000e+00,1.210000e+01,1.000000e+00,5.000000e-01,...,0.000000e+00,1.000000e+00,2.025000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2025-01-16 00:00:00,0 days 15:32:57,2025-01-16 00:00:00,0 days 15:43:00
75%,2.000000e+00,1.000000e+00,2.980000e+00,1.000000e+00,2.340000e+02,2.350000e+02,1.000000e+00,1.910000e+01,2.500000e+00,5.000000e-01,...,0.000000e+00,1.000000e+00,2.855000e+01,2.500000e+00,0.000000e+00,7.500000e-01,2025-01-24 00:00:00,0 days 19:05:06,2025-01-24 00:00:00,0 days 19:12:45
max,7.000000e+00,9.000000e+00,4.473030e+04,9.900000e+01,2.650000e+02,2.650000e+02,5.000000e+00,8.633721e+05,1.500000e+01,6.500000e+00,...,1.709400e+02,1.000000e+00,8.633804e+05,2.500000e+00,6.750000e+00,7.500000e-01,2025-02-01 00:00:00,0 days 23:59:59,2025-02-01 00:00:00,0 days 23:59:59
std,4.348649e-01,7.505595e-01,4.301774e+01,1.173975e+01,6.301632e+01,6.885694e+01,4.990562e-01,5.090525e+02,1.902679e+00,6.627530e-02,...,2.094945e+00,1.372759e-01,5.092210e+02,6.743995e-01,4.641857e-01,3.589569e-01,NaN,0 days 05:32:03.289185,NaN,0 days 05:37:13.891896


Now we will create the staging table using this data.

In [28]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Database Connection Settings (from your Step 2/3 settings)
DB_USER = "postgres"
DB_PASSWORD = "Nisarg123"  # Replace with your actual password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Taxi_Database"  # Replace with your database name

# Create the SQLAlchemy engine for PostgreSQL
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# 2. Load your raw data into a Pandas DataFrame
# (This could be a CSV, JSON, or Excel file)

# 3. Clean column names (Optional but highly recommended for PostgreSQL)
# This removes spaces, forces lowercase, and replaces special characters
new_df.columns = new_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^a-zA-Z0-9_]', '', regex=True)

# 4. Define the staging table name
staging_table_name = "stg_customers"

# 5. Create table and load data
# if_exists='replace' drops the old staging table if it exists and creates 
# a new one matching the exact columns and data types of your DataFrame.
try:
    new_df.to_sql(
        name=staging_table_name,
        con=engine,
        if_exists="replace",  # Alternatives: 'fail' or 'append'
        index=False,          # Do not write the DataFrame index as a column
        chunksize=1000        # Loads data in batches for better performance
    )
    print(f"Successfully created staging table '{staging_table_name}' and loaded {len(df)} rows.")

except Exception as e:
    print(f"An error occurred: {e}")


C:\Users\imnis\AppData\Local\Temp\ipykernel_4540\889583173.py:29: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  new_df.to_sql(


Successfully created staging table 'stg_customers' and loaded 2935071 rows.


In [29]:
# ---------- 1. RESET DATABASE (Put this at the very top) ----------
# Connect to default 'postgres' database to drop and recreate your target database

import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = "postgres"
DB_PASSWORD = "Nisarg123"  # Replace with your actual password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Taxi_Database"  # Replace with your database name
# ---------- 1. RESET DATABASE (Updated for PostgreSQL Autocommit) ----------
# Connect to default 'postgres' database
admin_engine = create_engine(f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/postgres")

# Open a standard connection instead of a transaction block (.begin)
with admin_engine.connect() as connection:
    # Set isolation level to AUTOCOMMIT to bypass transaction block restrictions
    connection = connection.execution_options(isolation_level="AUTOCOMMIT")
    
    # 1. Kick out active connections so the database isn't locked
    connection.execute(text(f"""
        SELECT pg_terminate_backend(pg_stat_activity.pid)
        FROM pg_stat_activity
        WHERE pg_stat_activity.datname = '{DB_NAME}' AND pid <> pg_backend_pid();
    """))
    print("Terminated active connections.")
    
    # 2. Safely drop the old database without transaction errors
    connection.execute(text(f"DROP DATABASE IF EXISTS {DB_NAME};"))
    print(f"Dropped existing database '{DB_NAME}'.")
    
    # 3. Create the database fresh
    connection.execute(text(f"CREATE DATABASE {DB_NAME};"))
    print(f"Database '{DB_NAME}' created fresh from scratch.")

Terminated active connections.
Dropped existing database 'Taxi_Database'.
Database 'Taxi_Database' created fresh from scratch.


## Building the Dimension Tables

With the cleaned staging data loaded, we now build the four dimension tables for the star schema:

- **Dim_location** — one row per TLC `LocationID`, sourced from `taxi_zone_lookup.csv` (borough, zone, service zone). Joins to the fact via `PULocationID` / `DOLocationID`.
- **dim_payment** — decodes `payment_type` using the NYC TLC data dictionary.
- **dim_rate_code** — decodes `RatecodeID` using the NYC TLC data dictionary.
- **dim_date** — a continuous calendar covering every pickup/dropoff date, with `date_key` (YYYYMMDD) as the surrogate key.

Each dimension is then loaded into PostgreSQL alongside the staging table.

In [30]:
# ---------- Dim_location ----------
# Source of truth: taxi_zone_lookup.csv (one row per TLC LocationID).
from pathlib import Path

base_path = Path(r"C:\Taxi")
zone_data = pd.read_csv(base_path / "taxi_zone_lookup.csv")

dim_location = zone_data.rename(
    columns={
        "LocationID": "location_id",
        "Borough": "borough",
        "Zone": "zone",
        "service_zone": "service_zone",
    }
).copy()

# Tidy text and fill the handful of N/A rows (LocationID 264 & 265)
for col in ["borough", "zone", "service_zone"]:
    dim_location[col] = dim_location[col].astype("string").str.strip()
dim_location[["borough", "zone", "service_zone"]] = dim_location[
    ["borough", "zone", "service_zone"]
].fillna("Unknown")

dim_location = (
    dim_location.drop_duplicates("location_id")
    .sort_values("location_id")
    .reset_index(drop=True)
)

print("dim_location shape:", dim_location.shape)
dim_location.head()

dim_location shape: (265, 4)


,location_id,borough,zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [31]:
# ---------- dim_payment ----------
# NYC TLC data dictionary mapping for payment_type (0 = Flex Fare appears in 2025 data).
payment_map = {
    0: "Flex Fare trip",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip",
}

dim_payment = pd.DataFrame(
    sorted(payment_map.items()),
    columns=["payment_type", "payment_description"],
)
dim_payment

,payment_type,payment_description
0,0,Flex Fare trip
1,1,Credit card
2,2,Cash
3,3,No charge
4,4,Dispute
5,5,Unknown
6,6,Voided trip


In [32]:
# ---------- dim_rate_code ----------
# NYC TLC data dictionary mapping for RatecodeID (99 = system-coded Unknown).
rate_code_map = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride",
    99: "Unknown",
}

dim_rate_code = pd.DataFrame(
    sorted(rate_code_map.items()),
    columns=["rate_code_id", "rate_code_description"],
)
dim_rate_code

,rate_code_id,rate_code_description
0,1,Standard rate
1,2,JFK
2,3,Newark
3,4,Nassau or Westchester
4,5,Negotiated fare
5,6,Group ride
6,99,Unknown


In [34]:
# ---------- dim_date ----------
# A continuous calendar spanning every date in the trip data (pickup + dropoff).
# `new_df` may have either the original column names or the lowercase versions
# created later in the notebook, so resolve the right ones dynamically.

def resolve_column(df, *candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"None of the expected date columns were found: {candidates}")

pickup_date_col = resolve_column(new_df, "Pickup_date", "pickup_date")
dropoff_date_col = resolve_column(new_df, "Dropoff_date", "dropoff_date")

date_bounds = pd.concat([new_df[pickup_date_col], new_df[dropoff_date_col]])
start_date, end_date = date_bounds.min(), date_bounds.max()

calendar = pd.date_range(start=start_date, end=end_date, freq="D")

dim_date = pd.DataFrame({"full_date": calendar})
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)  # e.g. 20250101
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["day_of_week"] = dim_date["full_date"].dt.dayofweek + 1  # 1 = Monday ... 7 = Sunday
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek >= 5
dim_date["week_of_year"] = dim_date["full_date"].dt.isocalendar().week.astype(int)

# Put the surrogate key first for a tidy dimension layout
dim_date = dim_date[[
    "date_key", "full_date", "day", "month", "month_name", "quarter",
    "year", "day_of_week", "day_name", "is_weekend", "week_of_year",
]]

print("dim_date shape:", dim_date.shape)
dim_date.head()

dim_date shape: (33, 11)


,date_key,full_date,day,month,month_name,quarter,year,day_of_week,day_name,is_weekend,week_of_year
0,20241231,2024-12-31,31,12,December,4,2024,2,Tuesday,False,1
1,20250101,2025-01-01,1,1,January,1,2025,3,Wednesday,False,1
2,20250102,2025-01-02,2,1,January,1,2025,4,Thursday,False,1
3,20250103,2025-01-03,3,1,January,1,2025,5,Friday,False,1
4,20250104,2025-01-04,4,1,January,1,2025,6,Saturday,True,1


In [36]:
# ---------- Load all dimension tables into PostgreSQL ----------
# Uses the same `engine` created for the staging table above.
import pandas as pd
from sqlalchemy import create_engine, text

DB_USER = "postgres"
DB_PASSWORD = "Nisarg123"  # Replace with your actual password
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "Taxi_Database"  # Replace with your database name

# Create the SQLAlchemy engine for PostgreSQL
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)


dimensions = {
    "dim_location": dim_location,
    "dim_payment": dim_payment,
    "dim_rate_code": dim_rate_code,
    "dim_date": dim_date,
}

# Drop any existing star-schema tables first so re-runs do not fail on
# foreign-key dependencies from fact_trips.
with engine.begin() as conn:
    for table_name in ["fact_trips", *dimensions.keys()]:
        conn.execute(text(f"DROP TABLE IF EXISTS {table_name} CASCADE"))

for table_name, frame in dimensions.items():
    frame.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",  # drop + recreate so reruns stay idempotent
        index=False,
    )
    print(f"Loaded {table_name}: {len(frame)} rows")

Loaded dim_location: 265 rows
Loaded dim_payment: 7 rows
Loaded dim_rate_code: 7 rows
Loaded dim_date: 33 rows


## Building the Fact Table

`fact_trips` is the center of the star schema — one row per taxi trip. It carries **foreign keys** to each dimension plus the numeric **measures**:

- **Foreign keys** → `pickup_date_key` / `dropoff_date_key` (→ `dim_date`), `pickup_location_id` / `dropoff_location_id` (→ `dim_location`), `payment_type` (→ `dim_payment`), `rate_code_id` (→ `dim_rate_code`).
- **Degenerate dimensions** → `vendor_id`, `store_and_fwd_flag`, `pickup_time`, `dropoff_time`.
- **Measures** → `passenger_count`, `trip_distance`, `fare_amount`, and all the fare/surcharge components through `total_amount`.

Because every dimension uses a natural key (or the deterministic `date_key`), the fact is built directly from `new_df` with no joins. It's then loaded into PostgreSQL, and finally primary/foreign-key constraints are applied to wire the star schema together.

In [43]:
# ---------- fact_trips ----------
# One row per taxi trip, built from the cleaned `new_df`.
# Foreign keys reference the dimensions (all natural keys, so no lookups needed):
#   pickup/dropoff_date_key -> dim_date, *_location_id -> dim_location,
#   payment_type -> dim_payment, rate_code_id -> dim_rate_code.
def resolve_column(df, *candidates):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    raise KeyError(f"None of the expected columns were found: {candidates}")

vendor_col = resolve_column(new_df, "VendorID", "vendorid", "vendor_id")
pickup_date_col = resolve_column(new_df, "Pickup_date", "pickup_date")
dropoff_date_col = resolve_column(new_df, "Dropoff_date", "dropoff_date")
pickup_time_col = resolve_column(new_df, "Pickup_time", "pickup_time")
dropoff_time_col = resolve_column(new_df, "Dropoff_time", "dropoff_time")
pulocation_col = resolve_column(new_df, "PULocationID", "pulocationid", "pu_location_id")
dolocation_col = resolve_column(new_df, "DOLocationID", "dolocationid", "do_location_id")
ratecode_col = resolve_column(new_df, "RatecodeID", "ratecodeid", "rate_code_id")
payment_col = resolve_column(new_df, "payment_type", "paymenttype")
store_flag_col = resolve_column(new_df, "store_and_fwd_flag", "store_and_fwd_flag")
passenger_col = resolve_column(new_df, "passenger_count", "passenger_count")
trip_distance_col = resolve_column(new_df, "trip_distance", "trip_distance")
fare_amount_col = resolve_column(new_df, "fare_amount", "fare_amount")
extra_col = resolve_column(new_df, "extra", "extra")
mta_tax_col = resolve_column(new_df, "mta_tax", "mta_tax")
tip_amount_col = resolve_column(new_df, "tip_amount", "tip_amount")
tolls_amount_col = resolve_column(new_df, "tolls_amount", "tolls_amount")
improvement_surcharge_col = resolve_column(new_df, "improvement_surcharge", "improvement_surcharge")
congestion_surcharge_col = resolve_column(new_df, "congestion_surcharge", "congestion_surcharge")
airport_fee_col = resolve_column(new_df, "Airport_fee", "airport_fee")
cbd_congestion_fee_col = resolve_column(new_df, "cbd_congestion_fee", "cbd_congestion_fee")
total_amount_col = resolve_column(new_df, "total_amount", "total_amount")

fact_trips = pd.DataFrame({
    # --- dimension foreign keys ---
    "vendor_id": new_df[vendor_col].astype("int32"),
    "pickup_date_key": new_df[pickup_date_col].dt.strftime("%Y%m%d").astype("int32"),
    "dropoff_date_key": new_df[dropoff_date_col].dt.strftime("%Y%m%d").astype("int32"),
    "pickup_location_id": new_df[pulocation_col].astype("int32"),
    "dropoff_location_id": new_df[dolocation_col].astype("int32"),
    "rate_code_id": new_df[ratecode_col].astype("int32"),
    "payment_type": new_df[payment_col].astype("int32"),
    # --- degenerate dimensions / attributes ---
    "store_and_fwd_flag": new_df[store_flag_col].astype("string"),
    "pickup_time": new_df[pickup_time_col],
    "dropoff_time": new_df[dropoff_time_col],
    # --- measures ---
    "passenger_count": new_df[passenger_col].astype("int32"),
    "trip_distance": new_df[trip_distance_col],
    "fare_amount": new_df[fare_amount_col],
    "extra": new_df[extra_col],
    "mta_tax": new_df[mta_tax_col],
    "tip_amount": new_df[tip_amount_col],
    "tolls_amount": new_df[tolls_amount_col],
    "improvement_surcharge": new_df[improvement_surcharge_col],
    "congestion_surcharge": new_df[congestion_surcharge_col],
    "airport_fee": new_df[airport_fee_col],
    "cbd_congestion_fee": new_df[cbd_congestion_fee_col],
    "total_amount": new_df[total_amount_col],
})

print("fact_trips shape:", fact_trips.shape)
print("null cells:", int(fact_trips.isna().sum().sum()))
fact_trips.head()

fact_trips shape: (2879944, 22)
null cells: 0


,vendor_id,pickup_date_key,dropoff_date_key,pickup_location_id,dropoff_location_id,rate_code_id,payment_type,store_and_fwd_flag,pickup_time,dropoff_time,...,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,airport_fee,cbd_congestion_fee,total_amount
0,1,20250101,20250101,229,237,1,1,N,0 days 00:18:38,0 days 00:26:59,...,10.0,3.5,0.5,3.00,0.0,1.0,2.5,0.0,0.0,18.00
1,1,20250101,20250101,236,237,1,1,N,0 days 00:32:40,0 days 00:35:13,...,5.1,3.5,0.5,2.02,0.0,1.0,2.5,0.0,0.0,12.12
2,1,20250101,20250101,141,141,1,1,N,0 days 00:44:04,0 days 00:46:01,...,5.1,3.5,0.5,2.00,0.0,1.0,2.5,0.0,0.0,12.10
3,2,20250101,20250101,244,244,1,2,N,0 days 00:14:27,0 days 00:20:01,...,7.2,1.0,0.5,0.00,0.0,1.0,0.0,0.0,0.0,9.70
4,2,20250101,20250101,244,116,1,2,N,0 days 00:21:34,0 days 00:25:06,...,5.8,1.0,0.5,0.00,0.0,1.0,0.0,0.0,0.0,8.30


In [44]:
# ---------- Load fact_trips into PostgreSQL ----------
# ~2.9M rows -> loaded in chunks; this may take a few minutes (same as the staging load).
#
# pickup_time / dropoff_time are timedeltas, which pandas cannot map to a Postgres
# type -- left alone it silently writes the raw int64 (microseconds since midnight)
# into a bigint column. Convert to datetime.time and pin the column type so the
# fact table gets real TIME columns.
from sqlalchemy.types import Time

for c in ["pickup_time", "dropoff_time"]:
    if pd.api.types.is_timedelta64_dtype(fact_trips[c]):  # guard: cell is safe to re-run
        fact_trips[c] = (pd.Timestamp("1970-01-01") + fact_trips[c]).dt.time

fact_table_name = "fact_trips"

fact_trips.to_sql(
    name=fact_table_name,
    con=engine,
    if_exists="replace",  # drop + recreate so reruns stay idempotent
    index=False,
    chunksize=10000,
    dtype={"pickup_time": Time(), "dropoff_time": Time()},
)
print(f"Loaded {fact_table_name}: {len(fact_trips)} rows")

Loaded fact_trips: 2879944 rows


In [ ]:
# ---------- Enforce star-schema keys & referential integrity ----------
# Adds primary keys to the dimensions, a surrogate key + foreign keys to the fact.
# Each statement runs independently so re-running the cell just skips what already exists.
from sqlalchemy import text

statements = [
    # Dimension primary keys
    "ALTER TABLE dim_location  ADD PRIMARY KEY (location_id)",
    "ALTER TABLE dim_payment   ADD PRIMARY KEY (payment_type)",
    "ALTER TABLE dim_rate_code ADD PRIMARY KEY (rate_code_id)",
    "ALTER TABLE dim_date      ADD PRIMARY KEY (date_key)",
    # Fact surrogate key
    "ALTER TABLE fact_trips ADD COLUMN trip_id BIGSERIAL PRIMARY KEY",
    # Fact foreign keys -> dimensions
    "ALTER TABLE fact_trips ADD CONSTRAINT fk_pickup_location  FOREIGN KEY (pickup_location_id)  REFERENCES dim_location(location_id)",
    "ALTER TABLE fact_trips ADD CONSTRAINT fk_dropoff_location FOREIGN KEY (dropoff_location_id) REFERENCES dim_location(location_id)",
    "ALTER TABLE fact_trips ADD CONSTRAINT fk_payment          FOREIGN KEY (payment_type)        REFERENCES dim_payment(payment_type)",
    "ALTER TABLE fact_trips ADD CONSTRAINT fk_rate_code        FOREIGN KEY (rate_code_id)        REFERENCES dim_rate_code(rate_code_id)",
    "ALTER TABLE fact_trips ADD CONSTRAINT fk_pickup_date      FOREIGN KEY (pickup_date_key)     REFERENCES dim_date(date_key)",
    "ALTER TABLE fact_trips ADD CONSTRAINT fk_dropoff_date     FOREIGN KEY (dropoff_date_key)    REFERENCES dim_date(date_key)",
]

with engine.connect() as conn:
    for stmt in statements:
        try:
            conn.execute(text(stmt))
            conn.commit()
            print("OK      :", stmt[:70])
        except Exception as e:
            conn.rollback()
            if "multiple primary keys" in str(e).lower() or "already exists" in str(e).lower():
                print("Already present:", stmt[:70])
            else:
                print("Skipped :", stmt[:70], "->", str(e).splitlines()[0][:60])

Skipped : ALTER TABLE dim_location  ADD PRIMARY KEY (location_id) -> (psycopg2.errors.InvalidTableDefinition) multiple primary ke
Skipped : ALTER TABLE dim_payment   ADD PRIMARY KEY (payment_type) -> (psycopg2.errors.InvalidTableDefinition) multiple primary ke
Skipped : ALTER TABLE dim_rate_code ADD PRIMARY KEY (rate_code_id) -> (psycopg2.errors.InvalidTableDefinition) multiple primary ke
Skipped : ALTER TABLE dim_date      ADD PRIMARY KEY (date_key) -> (psycopg2.errors.InvalidTableDefinition) multiple primary ke
OK      : ALTER TABLE fact_trips ADD COLUMN trip_id BIGSERIAL PRIMARY KEY
OK      : ALTER TABLE fact_trips ADD CONSTRAINT fk_pickup_location  FOREIGN KEY 
OK      : ALTER TABLE fact_trips ADD CONSTRAINT fk_dropoff_location FOREIGN KEY 
OK      : ALTER TABLE fact_trips ADD CONSTRAINT fk_payment          FOREIGN KEY 
OK      : ALTER TABLE fact_trips ADD CONSTRAINT fk_rate_code        FOREIGN KEY 
OK      : ALTER TABLE fact_trips ADD CONSTRAINT fk_pickup_date      FOREIGN KEY 
OK